# SINDy Dengue: Paper Analysis Notebook
## V1 (Delayed SIR) & V2 (Delayed SEIR)

Three paper-essential components:
1. **Forward simulation** — integrate recovered SINDy equations, compare to held-out data
2. **Structural comparison** — quantify similarity to canonical SIR/SEIR
3. **Bootstrap confidence intervals** — uncertainty on recovered coefficients

> Set `DATA_PATH` in Section 0 to use real data. Otherwise synthetic data is generated.

## 0. Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import UnivariateSpline
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.optimize import minimize
from itertools import combinations_with_replacement, product as iproduct
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── USER SETTINGS ─────────────────────────────────────────────────────────
DATA_PATH  = None       # set to 'your_file.csv'
Nh         = 500_000    # total population
DELTA      = 7/4        # incubation rate (week^-1)
GAMMA      = 1.0        # recovery rate (week^-1)
MU_H       = 1/(70*52)  # natural death rate (week^-1)
SIGMA      = 1.5        # Gaussian smoothing sigma
THRESHOLD  = 0.05       # STLS sparsity threshold
POLY_ORDER = 2          # library polynomial order
N_WEEKS    = 312        # weeks for synthetic data
HOLDOUT    = 52         # weeks held out for forward simulation test
N_BOOT     = 500        # bootstrap resamples

TAU_GRID   = [1, 2, 3, 4]   # V1 delay grid
TAU1_GRID  = [1, 2, 3]      # V2 incubation delay grid
TAU2_GRID  = [1, 2, 3]      # V2 recovery delay grid

print('Config set.')

## 1. Data

In [ ]:
def generate_synthetic(n_weeks=312, seed=42, noise=0.15):
    rng = np.random.default_rng(seed)
    _Nh=362_202; _Ah=7233/52; _mu=_Ah/_Nh
    _d=7/4; _g=1.0; _muv=0.5; _Nv=_Nh
    t   = np.arange(n_weeks, dtype=float)
    bh  = 0.45 + 0.08*np.sin(2*np.pi*t/52) + 0.05*np.sin(4*np.pi*t/52+0.5)
    def rhs(s, b):
        Sh,Eh,Ih,Rh = s
        f = b*2.5*_Nv*Ih*Sh/((b*2.5*Ih+_muv*_Nh)*_muv*_Nh)
        return np.array([_Ah-f-_mu*Sh, f-(_d+_mu)*Eh,
                         _d*Eh-(_g+_mu)*Ih, _g*Ih-_mu*Rh])
    st=np.array([_Nh-13,5.,13.,0.]); Iv=np.zeros(n_weeks)
    for i in range(n_weeks):
        k1=rhs(st,bh[i]); k2=rhs(st+.5*k1,bh[i])
        k3=rhs(st+.5*k2,bh[i]); k4=rhs(st+k3,bh[i])
        st=np.maximum(st+(k1+2*k2+2*k3+k4)/6,0); Iv[i]=st[2]
    return t, rng.poisson(np.maximum(Iv*(1+noise*rng.standard_normal(n_weeks)),0.5)).astype(float)

if DATA_PATH:
    import pandas as pd
    I_all = pd.read_csv(DATA_PATH, header=0).iloc[:,0].values.astype(float)
    t_all = np.arange(len(I_all), dtype=float)
else:
    print('No DATA_PATH — using synthetic placeholder.')
    t_all, I_all = generate_synthetic(N_WEEKS)

t_train, I_train = t_all[:-HOLDOUT], I_all[:-HOLDOUT]
t_test,  I_test  = t_all[-HOLDOUT:], I_all[-HOLDOUT:]
spl = UnivariateSpline(t_train, I_train,
                       s=len(t_train)*np.var(I_train)*0.1, k=4)
I_smooth = spl(t_train)
print(f'Train: {len(t_train)} weeks | Hold-out: {len(t_test)} weeks')

fig, ax = plt.subplots(figsize=(13,3))
ax.bar(t_all, I_all, color='steelblue', alpha=0.4, label='Raw')
ax.plot(t_train, I_smooth, 'r-', lw=2, label='Smoothed (train)')
ax.axvline(t_train[-1], color='k', ls='--', lw=1.2, label='Hold-out start')
ax.set_xlabel('Week'); ax.set_ylabel('Cases')
ax.set_title('Weekly Dengue Hospitalized Cases'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. SINDy Core

In [ ]:
def build_library(X, poly_order=2):
    n, m = X.shape
    cols = [np.ones(n)] + [X[:,i] for i in range(m)]
    if poly_order >= 2:
        for i,j in combinations_with_replacement(range(m), 2):
            cols.append(X[:,i]*X[:,j])
    return np.column_stack(cols)

def make_names(state_names, poly_order=2):
    m = len(state_names)
    names = ['1'] + list(state_names)
    if poly_order >= 2:
        for i,j in combinations_with_replacement(range(m), 2):
            names.append(state_names[i] + '*' + state_names[j])
    return names

def deriv(t, X, sigma=SIGMA):
    dX = np.gradient(X, t, axis=0)
    for j in range(X.shape[1]):
        dX[:,j] = gaussian_filter1d(dX[:,j], sigma=sigma)
    return dX

def stls(Theta, dX, threshold=THRESHOLD, max_iter=20):
    Theta = np.nan_to_num(Theta); dX = np.nan_to_num(dX)
    norms = np.linalg.norm(Theta, axis=0); norms[norms<1e-12] = 1.
    Tn    = Theta / norms
    Xi_n  = np.linalg.lstsq(Tn, dX, rcond=None)[0]
    for _ in range(max_iter):
        small = np.abs(Xi_n) < threshold; Xi_n[small] = 0.
        for j in range(dX.shape[1]):
            big = ~small[:,j]
            if big.sum() == 0: continue
            Xi_n[big,j] = np.linalg.lstsq(Tn[:,big], dX[:,j], rcond=None)[0]
    return Xi_n / norms[:,None]

def aic_bic(Theta, dX, Xi):
    n = Theta.shape[0]; dXh = Theta@Xi; rss = np.sum((dX-dXh)**2)
    s2 = max(rss/(n*dX.shape[1]), 1e-12)
    ll = -0.5*n*dX.shape[1]*(np.log(2*np.pi*s2)+1)
    k  = np.sum(Xi != 0)
    return 2*k-2*ll, k*np.log(n)-2*ll

def clip_pos(x, lbl=''):
    n = np.sum(x < 0)
    if n: print(f'  [clip] {lbl}: {n} neg to 0')
    return np.maximum(x, 0)

print('SINDy core defined.')

## 3. Compartment Builders

In [ ]:
def build_V1(I, t, tau=1):
    n = len(I); R = np.zeros(n)
    R[tau:] = I[:-tau] if tau > 0 else I
    S = clip_pos(Nh - I - R, 'V1 S')
    return t[tau:], np.column_stack([S, I, R])[tau:], ['S_tilde','I','R_tilde']

def build_V2(I, t, tau1=1, tau2=1):
    n = len(I); dt = t[1]-t[0]
    E = np.zeros(n); E[tau1:] = I[:-tau1] if tau1 > 0 else I
    R = np.zeros(n)
    for k in range(1, n):
        R[k] = R[k-1] + dt*(GAMMA*I[k-1] - MU_H*R[k-1])
    R = clip_pos(R, 'V2 R'); S = clip_pos(Nh-E-I-R, 'V2 S')
    tm = max(tau1, tau2)
    return t[tm:], np.column_stack([S,E,I,R])[tm:], ['S','E','I','R']

def grid_search(build_fn, param_grid, verbose=True):
    keys = list(param_grid.keys()); results = []
    for combo in iproduct(*param_grid.values()):
        params = dict(zip(keys, combo))
        try:
            t_, X_, sn_ = build_fn(**params)
            Th_ = build_library(X_, POLY_ORDER)
            dX_ = deriv(t_, X_)
            Xi_ = stls(Th_, dX_)
            a, b = aic_bic(Th_, dX_, Xi_)
            results.append(dict(params=params, Xi=Xi_, t=t_, X=X_,
                                state_names=sn_, Theta=Th_, dX=dX_, aic=a, bic=b))
            if verbose:
                ps = ', '.join(f'{k}={v}' for k,v in params.items())
                print(f'  [{ps}]  AIC={a:.1f}  nz={np.sum(Xi_!=0)}')
        except Exception as e:
            if verbose: print(f'  {params} FAILED: {e}')
    results.sort(key=lambda r: r['aic'])
    return results[0], results

print('--- V1 grid search ---')
best_v1, all_v1 = grid_search(
    lambda tau: build_V1(I_smooth, t_train, tau=tau), {'tau': TAU_GRID})
t1,X1,sn1 = best_v1['t'],best_v1['X'],best_v1['state_names']
Th1,dX1,Xi1 = best_v1['Theta'],best_v1['dX'],best_v1['Xi']
aic1,bic1 = best_v1['aic'],best_v1['bic']
fn1 = make_names(sn1); tau_v1 = best_v1['params']['tau']

print('--- V2 grid search ---')
best_v2, all_v2 = grid_search(
    lambda tau1,tau2: build_V2(I_smooth,t_train,tau1=tau1,tau2=tau2),
    {'tau1': TAU1_GRID, 'tau2': TAU2_GRID})
t2,X2,sn2 = best_v2['t'],best_v2['X'],best_v2['state_names']
Th2,dX2,Xi2 = best_v2['Theta'],best_v2['dX'],best_v2['Xi']
aic2,bic2 = best_v2['aic'],best_v2['bic']
fn2 = make_names(sn2); tau1_v2,tau2_v2 = best_v2['params']['tau1'],best_v2['params']['tau2']

print(f'V1 best tau={tau_v1}  AIC={aic1:.1f}')
print(f'V2 best tau1={tau1_v2}, tau2={tau2_v2}  AIC={aic2:.1f}')

## 4. Forward Simulation

Integrate the recovered SINDy ODE forward from the last training state.
Compare against held-out test data.

**Canonical SIR:** dS/dt = -beta*S*I, dI/dt = beta*S*I - gamma*I, dR/dt = gamma*I

**Canonical SEIR:** dS/dt = -beta*S*I, dE/dt = beta*S*I - sigma*E, dI/dt = sigma*E - gamma*I, dR/dt = gamma*I

In [ ]:
def sindy_rhs(t_val, state, Xi, poly_order=POLY_ORDER):
    X_pt  = np.array(state).reshape(1, -1)
    Theta = build_library(X_pt, poly_order)
    return (Theta @ Xi).flatten()

def forward_simulate(Xi, X_init, t_start, t_end, n_eval=500):
    t_eval = np.linspace(t_start, t_end, n_eval)
    sol = solve_ivp(
        lambda t, y: sindy_rhs(t, y, Xi),
        (t_start, t_end), X_init, t_eval=t_eval,
        method='RK45', max_step=0.5, rtol=1e-6, atol=1e-8)
    return sol.t, np.maximum(sol.y.T, 0)

def rmse(a,b): return np.sqrt(np.mean((a-b)**2))
def mae(a,b):  return np.mean(np.abs(a-b))
def r2(a,b):
    return 1 - np.sum((a-b)**2)/(np.sum((a-np.mean(a))**2)+1e-12)

# Simulate
t_sim1, X_sim1 = forward_simulate(Xi1, X1[-1], t1[-1], t_test[-1])
t_sim2, X_sim2 = forward_simulate(Xi2, X2[-1], t2[-1], t_test[-1])

I_idx1 = sn1.index('I'); I_idx2 = sn2.index('I')
I_pred1 = np.interp(t_test, t_sim1, X_sim1[:,I_idx1])
I_pred2 = np.interp(t_test, t_sim2, X_sim2[:,I_idx2])

for name, pred in [('V1', I_pred1), ('V2', I_pred2)]:
    print(f'{name}: RMSE={rmse(I_test,pred):.3f}  MAE={mae(I_test,pred):.3f}  R2={r2(I_test,pred):.3f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)
data = [
    (axes[0],'V1 Delayed SIR', t1, X1[:,I_idx1], t_sim1, X_sim1[:,I_idx1],
     I_pred1, 'steelblue'),
    (axes[1],'V2 Delayed SEIR',t2, X2[:,I_idx2], t_sim2, X_sim2[:,I_idx2],
     I_pred2, 'darkorange'),
]
for ax,name,t_tr,I_tr,t_sim,I_sim,I_pred,col in data:
    ax.scatter(t_all[:-HOLDOUT], I_all[:-HOLDOUT],
               s=8, color='gray', alpha=0.4, label='Train data')
    ax.scatter(t_test, I_test, s=14, color='black', zorder=5,
               label='Hold-out data')
    ax.plot(t_tr, I_tr, color=col, lw=2, label='SINDy training states')
    ax.plot(t_sim, I_sim, 'r--', lw=2, label='SINDy simulation')
    ax.axvline(t_tr[-1], color='k', lw=1, ls=':', alpha=0.6)
    ax.set_title(f'{name}  |  RMSE={rmse(I_test,I_pred):.2f}  R2={r2(I_test,I_pred):.3f}')
    ax.set_ylabel('I(t)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[-1].set_xlabel('Week')
plt.suptitle('Forward Simulation vs Hold-out Data', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Structural Comparison to Canonical SIR / SEIR

Three metrics:
1. **Dominant term analysis** — largest coefficients per equation
2. **Structural score** — fraction of coefficient mass on canonically expected terms
3. **Sign consistency** — do signs match epidemiological expectations

In [ ]:
# Expected terms per equation (substrings to match in feature names)
CANONICAL_V1 = {
    0: ['S_tilde*I', 'S_tilde'],  # dS_tilde/dt: -beta*S*I
    1: ['S_tilde*I', 'I*I', 'I'], # dI/dt: +beta*S*I - gamma*I
    2: ['I'],                      # dR_tilde/dt: +gamma*I
}
CANONICAL_V2 = {
    0: ['S*I', 'S'],  # dS/dt
    1: ['S*I', 'E'],  # dE/dt
    2: ['E', 'I'],    # dI/dt
    3: ['I', 'R'],    # dR/dt
}

SIGN_CHECK_V1 = [
    (0, 'S_tilde*I', -1, 'dS_tilde/dt: -beta*S*I removes susceptibles'),
    (1, 'S_tilde*I', +1, 'dI/dt:       +beta*S*I creates infectives'),
    (2, 'I',         +1, 'dR_tilde/dt: +gamma*I fills recovered'),
]
SIGN_CHECK_V2 = [
    (0, 'S*I', -1, 'dS/dt:  -beta*S*I'),
    (1, 'S*I', +1, 'dE/dt:  +beta*S*I'),
    (1, 'E*E', -1, 'dE/dt:  -sigma*E (look for E terms)'),
    (2, 'E',   +1, 'dI/dt:  +sigma*E'),
    (2, 'I*I', -1, 'dI/dt:  -gamma*I (look for I terms)'),
    (3, 'I',   +1, 'dR/dt:  +gamma*I'),
]

def dominant_terms(Xi, feat_names, state_names, top_n=5):
    print('Dominant terms:')
    for j, sn in enumerate(state_names):
        terms = sorted([(abs(Xi[i,j]), Xi[i,j], feat_names[i])
                        for i in range(len(feat_names)) if abs(Xi[i,j])>1e-10],
                       reverse=True)
        print(f'  d{sn}/dt:')
        for mag,val,nm in terms[:top_n]:
            print(f'    {val:+.5f}  {nm}')

def structural_score(Xi, feat_names, canonical_map):
    scores = {}
    for j, subs in canonical_map.items():
        total = np.sum(np.abs(Xi[:,j]))
        if total < 1e-12: scores[j]=0.; continue
        expected = sum(abs(Xi[i,j]) for i,fn in enumerate(feat_names)
                       if any(s in fn for s in subs))
        scores[j] = expected/total
    return scores, np.mean(list(scores.values()))

def sign_check(Xi, feat_names, checks):
    print('Sign consistency:')
    for eq_j, substr, expected_sign, desc in checks:
        matches = [(i, Xi[i,eq_j]) for i,fn in enumerate(feat_names)
                   if substr in fn and abs(Xi[i,eq_j])>1e-10]
        if not matches:
            print(f'  [NOT FOUND]  {desc}')
        else:
            best = max(matches, key=lambda x: abs(x[1]))
            ok = np.sign(best[1]) == expected_sign
            status = 'OK' if ok else 'WRONG SIGN'
            print(f'  [{status:<10}]  {desc}  (coef={best[1]:+.4f})')

print('=== V1 ===')
dominant_terms(Xi1, fn1, sn1)
sc1, overall1 = structural_score(Xi1, fn1, CANONICAL_V1)
print(f'Structural score: {overall1:.3f}')
sign_check(Xi1, fn1, SIGN_CHECK_V1)

print('=== V2 ===')
dominant_terms(Xi2, fn2, sn2)
sc2, overall2 = structural_score(Xi2, fn2, CANONICAL_V2)
print(f'Structural score: {overall2:.3f}')
sign_check(Xi2, fn2, SIGN_CHECK_V2)

In [ ]:
# ── Coefficient heatmap with canonical term highlights ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, Xi, fn, sn, title, canonical in [
    (axes[0], Xi1, fn1, sn1, 'V1: Delayed SIR', CANONICAL_V1),
    (axes[1], Xi2, fn2, sn2, 'V2: Delayed SEIR', CANONICAL_V2),
]:
    vm = max(np.abs(Xi).max(), 1e-6)
    im = ax.imshow(Xi, aspect='auto', cmap='RdBu_r', vmin=-vm, vmax=vm)
    ax.set_xticks(range(len(sn)))
    ax.set_xticklabels([f'd{s}/dt' for s in sn], rotation=40, ha='right')
    ax.set_yticks(range(len(fn)))
    ax.set_yticklabels(fn, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.03)
    for j, subs in canonical.items():
        for i, fname in enumerate(fn):
            if any(s in fname for s in subs) and abs(Xi[i,j])>1e-10:
                ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,
                    fill=False, edgecolor='lime', lw=2.5))
    ax.set_title(title)
fig.suptitle('Coefficient Matrix Xi  (green = canonically expected terms)', fontsize=12)
plt.tight_layout(); plt.show()

## 6. Bootstrap Confidence Intervals

Resample derivative residuals, re-run STLS, collect percentile CIs.
Also reports **selection frequency** — how often each term is chosen across resamples.

In [ ]:
def bootstrap_sindy(Theta, dX, Xi_ref, n_boot=N_BOOT, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    n   = Theta.shape[0]
    res = dX - Theta @ Xi_ref
    boot = []
    for b in range(n_boot):
        idx    = rng.integers(0, n, size=n)
        dX_b   = Theta @ Xi_ref + res[idx]
        try: boot.append(stls(Theta, dX_b))
        except: pass
        if verbose and (b+1) % 100 == 0:
            print(f'  Bootstrap {b+1}/{n_boot}')
    boot = np.array(boot)
    return boot, np.percentile(boot,2.5,axis=0), np.percentile(boot,97.5,axis=0)

print('Bootstrap V1...')
boot1, lo1, hi1 = bootstrap_sindy(Th1, dX1, Xi1)
print('Bootstrap V2...')
boot2, lo2, hi2 = bootstrap_sindy(Th2, dX2, Xi2)
print('Done.')

In [ ]:
def plot_bootstrap_ci(Xi, lo, hi, boot, feat_names, state_names, title):
    n_eq = len(state_names)
    fig, axes = plt.subplots(1, n_eq, figsize=(4*n_eq, 5))
    if n_eq == 1: axes = [axes]
    colors = ['steelblue','darkorange','crimson','forestgreen','purple']
    for j,(ax,sn) in enumerate(zip(axes,state_names)):
        nz = [i for i in range(len(feat_names)) if abs(Xi[i,j])>1e-8]
        if not nz: ax.set_title(f'd{sn}/dt = 0'); continue
        vals = Xi[nz,j]; lo_ = lo[nz,j]; hi_ = hi[nz,j]
        names = [feat_names[i] for i in nz]
        y = np.arange(len(nz))
        ax.barh(y, vals, xerr=[vals-lo_, hi_-vals],
                color=colors[j%len(colors)], alpha=0.7,
                error_kw=dict(ecolor='black', capsize=4, lw=1.5))
        ax.axvline(0, color='k', lw=0.8, ls='--')
        ax.set_yticks(y); ax.set_yticklabels(names, fontsize=8)
        ax.set_title(f'd{sn}/dt'); ax.set_xlabel('Coefficient')
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(f'{title}  (95% Bootstrap CI, n={N_BOOT})', fontsize=12)
    plt.tight_layout(); plt.show()

plot_bootstrap_ci(Xi1, lo1, hi1, boot1, fn1, sn1, 'V1 Delayed SIR')
plot_bootstrap_ci(Xi2, lo2, hi2, boot2, fn2, sn2, 'V2 Delayed SEIR')

In [ ]:
def selection_freq(boot, feat_names, state_names, thr=1e-8):
    freq = np.mean(np.abs(boot) > thr, axis=0)
    print('Selection frequency (stable coefs ~ 1.0):')
    for j, sn in enumerate(state_names):
        sel = [(freq[i,j], feat_names[i])
               for i in range(len(feat_names)) if freq[i,j]>0.1]
        if not sel: continue
        print(f'  d{sn}/dt:')
        for f, nm in sorted(sel, reverse=True):
            bar = '#'*int(f*20) + '.'*int((1-f)*20)
            print(f'    [{bar}] {f:.2f}  {nm}')

print('=== V1 Selection Frequency ===')
selection_freq(boot1, fn1, sn1)
print('=== V2 Selection Frequency ===')
selection_freq(boot2, fn2, sn2)

## 7. Delay Sensitivity Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# V1: AIC vs tau
taus = [r['params']['tau'] for r in all_v1]
aics = [r['aic'] for r in all_v1]
order = np.argsort(taus)
axes[0].plot(np.array(taus)[order], np.array(aics)[order],
             'o-', color='steelblue', lw=2, ms=8)
axes[0].axvline(tau_v1, color='red', ls='--', lw=1.5, label=f'Best tau={tau_v1}')
axes[0].set_xlabel('tau (weeks)'); axes[0].set_ylabel('AIC')
axes[0].set_title('V1: AIC vs Delay tau'); axes[0].legend()
axes[0].grid(alpha=0.3)

# V2: AIC surface
tau1s = sorted(set(r['params']['tau1'] for r in all_v2))
tau2s = sorted(set(r['params']['tau2'] for r in all_v2))
Z = np.full((len(tau1s), len(tau2s)), np.nan)
for r in all_v2:
    i=tau1s.index(r['params']['tau1']); j=tau2s.index(r['params']['tau2'])
    Z[i,j]=r['aic']
im = axes[1].imshow(Z, aspect='auto', cmap='viridis', origin='lower')
axes[1].set_xticks(range(len(tau2s))); axes[1].set_xticklabels(tau2s)
axes[1].set_yticks(range(len(tau1s))); axes[1].set_yticklabels(tau1s)
axes[1].set_xlabel('tau2'); axes[1].set_ylabel('tau1')
axes[1].set_title('V2: AIC Surface (tau1 x tau2)')
bi=tau1s.index(tau1_v2); bj=tau2s.index(tau2_v2)
axes[1].plot(bj, bi, 'r*', ms=16, label=f'Best (tau1={tau1_v2},tau2={tau2_v2})')
axes[1].legend(fontsize=8)
plt.colorbar(im, ax=axes[1], label='AIC')
plt.tight_layout(); plt.show()

## 8. Paper Summary Table

In [ ]:
rows = [
    ('V1 Delayed SIR',  f'tau={tau_v1}w',                  aic1,bic1,
     int(np.sum(Xi1!=0)), overall1, rmse(I_test,I_pred1), r2(I_test,I_pred1)),
    ('V2 Delayed SEIR', f'tau1={tau1_v2}w tau2={tau2_v2}w',aic2,bic2,
     int(np.sum(Xi2!=0)), overall2, rmse(I_test,I_pred2), r2(I_test,I_pred2)),
]
hdr = f"{'Variant':<20} {'Best tau':<20} {'AIC':>8} {'BIC':>8}"\
      f" {'nz':>4} {'Struct':>7} {'RMSE':>7} {'R2':>6}"
print('='*80)
print('PAPER SUMMARY TABLE'); print('='*80)
print(hdr); print('-'*80)
for name,tau,aic,bic,nz,st,rm,r2_ in rows:
    print(f'{name:<20} {tau:<20} {aic:>8.1f} {bic:>8.1f}'
          f' {nz:>4} {st:>7.3f} {rm:>7.3f} {r2_:>6.3f}')
print('='*80)
print('nz=nonzero coefficients | Struct=structural score vs canonical')
print('RMSE/R2 on hold-out test set')